# Dataset Preparation for the Fixed V1-V6 Pipeline

```
prepare_dataset_for_pipeline.py
-----------------------------------------------
PURPOSE
Your V1-V6 thesis notebooks are FIXED and must not be edited. They expect:
  - A CSV loaded via DATA_PATH
  - A target column named exactly what TARGET_COL is set to in Cell 1
    (currently 'program_stream')
  - No ID/junk columns (V1 drops a hardcoded COLS_TO_REMOVE list, but if
    the incoming CSV never contains those columns to begin with, that
    line is a harmless no-op)
  - Missing values represented as real NaN (not -9/-8/-7 etc. dataset-
    specific sentinel codes) so the notebooks' own dropna()/imputation
    logic works correctly regardless of which raw dataset it came from

This script is the ONE place that changes per dataset. You edit only the
CONFIG dict for a given experiment (exp1, exp2, ...) and run it. The
output is always a CSV shaped the same way: selected feature columns +
a target column literally named to match your notebook's TARGET_COL.
The notebooks themselves never change -- only DATA_PATH inside them
points to a different prepared CSV per experiment.

VALIDATION STEP (do this first)
Run this script with CONFIG_ORIGINAL below against your original
hybrid_student_performance_1200.csv. Since that dataset needs no
column selection, no missing-code recoding, and no renaming (the
target is already called 'program_stream'), the output CSV should be
IDENTICAL to the raw file. Feed that output into your unchanged V1
notebook and confirm you get the exact same results you already
documented. That confirms this script introduces no distortion before
you trust it on a new dataset.

Then switch to CONFIG_HSLS (or your own new CONFIG_expN) to prepare a
different dataset for the same unchanged pipeline.
```

## (Optional) Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np


### CONFIG 0 -- VALIDATION: your original dataset, should be a no-op

In [3]:
CONFIG_ORIGINAL = {
    "name": "original_synthetic_1200",
    "raw_path": "hybrid_student_performance_1200.csv",
    "output_path": "prepared_original_1200.csv",
    "target_col_raw": "program_stream",     # already matches TARGET_COL
    "target_col_out": "program_stream",     # no rename needed
    "keep_cols": None,                      # None = keep all columns as-is
    "drop_cols": [],                        # nothing to drop for validation run
    "missing_codes": [],                    # no sentinel codes in this dataset
    "codes_meaningful_not_missing": {},      # {col: [codes to KEEP as real categories]}
}


### CONFIG 1 -- HSLS:09, prepared for the SAME unchanged pipeline (exp1)

In [4]:
CONFIG_HSLS = {
    "name": "hsls09_exp1",
       "raw_path": "/content/drive/MyDrive/compinefile2/hsls09_16_student_pets_pear_v1_0.csv",   # your raw HSLS student file
    "output_path": "/content/drive/MyDrive/compinefile2/prepared_hsls_exp1.csv",
    "target_col_raw": "X4ENTRYMAJ23",
    "target_col_out": "program_stream",       # renamed to match TARGET_COL as-is
    "keep_cols": [
        # Demographic / SES
        "X1SEX", "X1RACE", "X1PAR1EDU",
        "X1MOMOCC2", "X1DADOCC2",
        "X1FAMINCOME", "X1SES", "X1SESQ5",
        # Academic Performance
        "X1TXMTSCOR", "X2TXMTSCOR",
        "X3TCREDAPMTH", "X3THIMATH", "X3TGPAMAT", "X3TGPAHIMTH",
        "X3TCREDAPSCI", "X3TCREDSCI", "X3THISCI", "X3THISCI9",
        "X3TGPASCI", "X3TGPAHISCI",
        "X3TCREDCOMPSCI", "X3TGPACOMPSCI",
        "X3TCREDCOM", "X3TGPACOM",
        "X3TCREDACAD", "X3TGPAACAD", "X3TGPASTEM", "X3TGPAMTHAP", "X3TGPASCIAP",
        "X3TAGPAWGT",
        # Career Interests
        "X1STUEDEXPCT",
        "S1MCAREER", "S1MUSEJOB", "S1SCAREER", "S1SUSEJOB",
        "S1MREASJOB", "S1SREASJOB", "S1PLAN",
        "S2JOBFAIR", "S2INTERN", "S2CAREERJOB", "S2CAREERINFLU",
        "S2MCAREER", "S2SCAREER", "S2HSPLAN", "S2MUSEJOB", "S2SUSEJOB",
        "C1JOBGUIDE",
        # Skills
        "S1MSKILLS", "S1SSKILLS", "S2MSKILLS", "S2SSKILLS",
        # Behavioral / Psychological
        "X1MTHID", "X1MTHUTI", "X1MTHEFF", "X1MTHINT",
        "X1SCIID", "X1SCIUTI", "X1SCIEFF", "X1SCIINT",
        "X1SCHOOLENG",
        "X2BEHAVEIN", "X2MEFFORT", "X2SEFFORT", "X2PROBLEM",
        "X2MTHID", "X2MTHEFF", "X2MTHINT_R", "X2SCIID", "X2SCIEFF",
        "X2S2SSPR12",
        # Target (raw name, handled separately below)
        "X4ENTRYMAJ23",
    ],
    "drop_cols": [],  # not needed -- keep_cols already whitelists exactly what we want
    "missing_codes": [-9, -8, -7, -6, -5, -4, -3],  # generic HSLS sentinel codes
    "codes_meaningful_not_missing": {
        "X4ENTRYMAJ23": [-1],  # -1 = "Undeclared/undecided", a real category
    },
}


### CONFIG 2 (continued)

In [5]:
# ==================================================================
# CONFIG 2 -- HSLS:09, using the audit notebook's evidence-based Set D
# (exp2) -- variable list read DYNAMICALLY from the decision table CSV,
# not hand-typed, so it can never drift out of sync with the audit.
# ==================================================================
DECISION_TABLE_PATH = "/content/drive/MyDrive/compinefile2/HSLS_final_predictor_decision_table.csv"

def load_set_d_variable_list(decision_table_path, target_col_raw):
    """Read the Variable column from the audit's final predictor decision
    table and return it as a keep_cols list (target appended, deduplicated,
    order preserved)."""
    decision_df = pd.read_csv(decision_table_path)
    variables = decision_df["Variable"].tolist()
    if target_col_raw not in variables:
        variables.append(target_col_raw)
    # de-duplicate while preserving order, in case the target already
    # appeared in the decision table for some reason
    seen = set()
    deduped = [v for v in variables if not (v in seen or seen.add(v))]
    return deduped

CONFIG_SET_D = {
    "name": "hsls09_setD",
     "raw_path": "/content/drive/MyDrive/compinefile2/hsls09_16_student_pets_pear_v1_0.csv",   # your raw HSLS student file -- same as CONFIG_HSLS
    "output_path": "/content/drive/MyDrive/compinefile2/prepared_hsls_setD.csv",
    "target_col_raw": "X4ENTRYMAJ23",
    "target_col_out": "program_stream",
    "keep_cols": load_set_d_variable_list(DECISION_TABLE_PATH, "X4ENTRYMAJ23"),
    "drop_cols": [],
    "missing_codes": [-9, -8, -7, -6, -5, -4, -3],   # same HSLS sentinel codes, unchanged
    "codes_meaningful_not_missing": {
        "X4ENTRYMAJ23": [-1],   # -1 = "Undeclared/undecided", unchanged
    },
}


# ==================================================================
# CONFIG 3 -- Same Set D predictor variables, but targeting the
# OFFICIAL NCES 10-category aggregation (X4ENTRYMAJ4Y) instead of the
# 23-category X4ENTRYMAJ23. Same underlying major-choice construct,
# same missingness pattern, same -1 "Undeclared" protection -- only the
# level of target granularity changes. This is NOT a researcher-built
# grouping: X4ENTRYMAJ4Y is an NCES-published aggregation used across
# their BPS, B&B, and NPSAS studies.
# ==================================================================
CONFIG_SET_D_4Y = {
    "name": "hsls09_setD_4Y",

    "raw_path":  "/content/drive/MyDrive/compinefile2/hsls09_16_student_pets_pear_v1_0.csv",   # your raw HSLS student file -- same as CONFIG_HSLS
    "output_path": "/content/drive/MyDrive/compinefile2/prepared_hsls_setD_4Y.csv",
    "target_col_raw": "X4ENTRYMAJ4Y",
    "target_col_out": "program_stream",
    # Same Set D predictor variables as before -- only the target changes.
    # Re-reads the same decision table; X4ENTRYMAJ4Y is appended as the
    # target instead of X4ENTRYMAJ23.
    "keep_cols": load_set_d_variable_list(DECISION_TABLE_PATH, "X4ENTRYMAJ4Y"),
    "drop_cols": [],
    "missing_codes": [-9, -8, -7, -6, -5, -4, -3],   # same HSLS sentinel codes, unchanged
    "codes_meaningful_not_missing": {
        "X4ENTRYMAJ4Y": [-1],   # -1 = "Undeclared/undecided", same protection as X4ENTRYMAJ23
    },
}


### CORE FUNCTIONS -- do not need to change between experiments

In [6]:
def load_raw(cfg: dict) -> pd.DataFrame:
    df = pd.read_csv(cfg["raw_path"], low_memory=False)
    print(f"[{cfg['name']}] Loaded raw shape: {df.shape}")
    return df


def select_columns(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if cfg["keep_cols"] is not None:
        keep = [c for c in cfg["keep_cols"] if c in df.columns]
        missing = set(cfg["keep_cols"]) - set(keep)
        if missing:
            print(f"[{cfg['name']}] WARNING: requested columns not found in raw file: {missing}")
        df = df[keep].copy()
    if cfg["drop_cols"]:
        present = [c for c in cfg["drop_cols"] if c in df.columns]
        df = df.drop(columns=present)
    return df


def recode_missing(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if not cfg["missing_codes"]:
        return df
    df = df.copy()
    meaningful = cfg.get("codes_meaningful_not_missing", {})
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        codes = list(cfg["missing_codes"])
        keep_codes = meaningful.get(col, [])
        codes = [c for c in codes if c not in keep_codes]
        if codes:
            df[col] = df[col].replace(codes, np.nan)
    return df


def rename_target(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    src, dst = cfg["target_col_raw"], cfg["target_col_out"]
    if src not in df.columns:
        raise ValueError(f"Target column '{src}' not found in dataframe.")
    if src != dst:
        df = df.rename(columns={src: dst})
    return df


def filter_valid_target(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    target = cfg["target_col_out"]
    before = len(df)
    df = df[df[target].notna()].copy()
    after = len(df)
    print(f"[{cfg['name']}] Rows with valid target: {before} -> {after} "
          f"({after/before:.1%} retained)")
    return df


def run(cfg: dict) -> pd.DataFrame:
    df = load_raw(cfg)
    df = select_columns(df, cfg)
    df = recode_missing(df, cfg)
    df = rename_target(df, cfg)
    df = filter_valid_target(df, cfg)

    df.to_csv(cfg["output_path"], index=False)
    print(f"[{cfg['name']}] Final shape: {df.shape}")
    print(f"[{cfg['name']}] Written to: {cfg['output_path']}")
    print(f"[{cfg['name']}] Target class distribution:")
    print(df[cfg["target_col_out"]].value_counts(dropna=False).sort_index())
    return df


## Run it
Pick the config for the experiment you want to run.

In [7]:
# run(CONFIG_ORIGINAL)
# run(CONFIG_HSLS)
# run(CONFIG_SET_D)

# NEW: same Set D predictors, official NCES 10-category target
run(CONFIG_SET_D_4Y)


[hsls09_setD_4Y] Loaded raw shape: (23503, 10521)
[hsls09_setD_4Y] Rows with valid target: 23503 -> 12829 (54.6% retained)
[hsls09_setD_4Y] Final shape: (12829, 378)
[hsls09_setD_4Y] Written to: /content/drive/MyDrive/compinefile2/prepared_hsls_setD_4Y.csv
[hsls09_setD_4Y] Target class distribution:
program_stream
-1.0      695
 1.0      493
 2.0     1166
 3.0     1645
 4.0      270
 5.0      966
 6.0      748
 7.0     2266
 8.0     1728
 9.0      760
 10.0    2092
Name: count, dtype: int64


,X1SEX,X2SEX,X3T1CREDPHYS,X2TXMQUINT,X2TXMTSCOR,X2TXMSCR,X3T1CREDCALC,X3T1CREDPREC,S2FAVSUBJ,S2SEX,...,S1MOMTALKS,S1NOTALKM,S1DADTALKM,S1MOMTALKM,S1WORKING,S1GETINTOCLG,S1PAYOFF,S1LEASTSUBJ,S1MOMTALKOTH,program_stream
1,2.0,2,0.0,4.0,54.0863,72.4904,0.0,0.0,1.0,2.0,...,1.0,0.0,1.0,1.0,3.0,4.0,4.0,NaN,1.0,9.0
2,2.0,2,0.0,4.0,55.6336,75.4243,0.0,1.0,1.0,2.0,...,1.0,0.0,0.0,1.0,4.0,2.0,2.0,7.0,1.0,5.0
6,2.0,2,0.0,2.0,47.6403,60.0290,0.0,0.0,9.0,2.0,...,0.0,0.0,NaN,1.0,3.0,3.0,3.0,11.0,1.0,7.0
7,1.0,1,0.0,5.0,62.8079,89.4800,0.0,1.0,8.0,1.0,...,1.0,0.0,1.0,1.0,2.0,4.0,4.0,1.0,1.0,8.0
8,1.0,1,0.0,5.0,59.8127,83.5930,0.0,0.0,13.0,1.0,...,0.0,0.0,0.0,1.0,1.0,3.0,2.0,1.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23495,1.0,1,1.0,5.0,69.8567,100.9180,1.0,1.0,3.0,1.0,...,0.0,1.0,0.0,0.0,3.0,3.0,3.0,1.0,0.0,3.0
23497,1.0,1,0.0,5.0,60.9403,85.8476,1.0,1.0,6.0,1.0,...,1.0,0.0,1.0,1.0,3.0,3.0,3.0,8.0,1.0,5.0
23500,2.0,2,0.0,3.0,48.1044,60.9564,0.0,1.0,11.0,2.0,...,0.0,1.0,0.0,0.0,3.0,3.0,3.0,3.0,0.0,7.0
23501,1.0,1,0.0,3.0,50.2191,65.1187,0.0,0.0,11.0,1.0,...,0.0,0.0,0.0,0.0,3.0,3.0,3.0,13.0,0.0,10.0
